# Wagmi SFT — Qwen2.5-1.5B-Instruct × Unsloth

Fine-tunes **Qwen/Qwen2.5-1.5B-Instruct** on the Deal ex Machina site content dataset using **Unsloth** + **TRL SFTTrainer**.

Target infra: HuggingFace Space with L40 GPU (48 GB VRAM).  
Expected runtime: ~15-25 min for 2 epochs on 374 training examples.

In [ ]:
!pip install "torch>=2.5.0"
!pip install "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install "transformers>=4.47.0" "datasets>=3.0.0" "trl>=0.12.0" "accelerate>=0.34.0" "peft>=0.14.0" "bitsandbytes>=0.45.0"
!pip install "sentencepiece>=0.2.0" "protobuf>=4.25.0" "huggingface_hub>=0.26.0"

In [ ]:
# Cell 2 — Imports & hyperparameters
# All tunable knobs live here — edit before running.

import os, json, torch
from pathlib import Path
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_ID       = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LEN    = 2048
DTYPE          = torch.bfloat16   # L40 supports bf16 natively
LOAD_IN_4BIT   = False            # 1.5B in bf16 ≈ 3 GB — no quantisation needed

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_R         = 32
LORA_ALPHA     = 64
LORA_DROPOUT   = 0.0              # 0 = Unsloth-optimised path
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# ── Training ───────────────────────────────────────────────────────────────
LEARNING_RATE     = 2e-4
NUM_EPOCHS        = 2
PER_DEVICE_BATCH  = 4             # baseline target: 4-8 on L40; 4 is safer
GRAD_ACCUM        = 2             # effective batch = 8
WARMUP_RATIO      = 0.05
LR_SCHEDULER      = "cosine"
WEIGHT_DECAY      = 0.01
MAX_GRAD_NORM     = 1.0
SAVE_STEPS        = 50
LOGGING_STEPS     = 10

# ── I/O ────────────────────────────────────────────────────────────────────
OUTPUT_DIR     = "wagmi-qwen2.5-1.5b-sft"
HUB_MODEL_ID   = "jeanbaptdzd/wagmi-qwen2.5-1.5b-sft"
PUSH_TO_HUB    = True
HF_TOKEN       = os.environ.get("HF_TOKEN")  # set in Space variables

# ── Logging ───────────────────────────────────────────────────────────────
# Switch to "wandb" only if WANDB_API_KEY is configured.
REPORT_TO      = "none"

# ── Paths ──────────────────────────────────────────────────────────────────
TRAIN_FILE = "data/train.jsonl"
EVAL_FILE  = "data/eval.jsonl"

print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")
print(json.dumps({
    "model": MODEL_ID, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
    "lr": LEARNING_RATE, "epochs": NUM_EPOCHS,
    "effective_batch": PER_DEVICE_BATCH * GRAD_ACCUM,
}, indent=2))

In [ ]:
# Cell 3 — Load base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_ID,
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = DTYPE,
    load_in_4bit    = LOAD_IN_4BIT,
)

# Qwen2.5 uses <|im_end|> as EOS; Unsloth handles this automatically.
# Ensure padding token is set (required by SFTTrainer).
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded. Parameters: {model.num_parameters() / 1e6:.1f}M")

In [ ]:
# Cell 4 — Attach LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r                   = LORA_R,
    lora_alpha          = LORA_ALPHA,
    lora_dropout        = LORA_DROPOUT,
    target_modules      = TARGET_MODULES,
    bias                = "none",
    use_gradient_checkpointing = "unsloth",  # Unsloth's memory-efficient checkpointing
    random_state        = 42,
    use_rslora          = False,
    loftq_config        = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable / 1e6:.2f}M / {total / 1e6:.1f}M ({100 * trainable / total:.2f}%)")

In [ ]:
# Cell 5 — Load and format dataset

raw = load_dataset(
    "json",
    data_files={"train": TRAIN_FILE, "eval": EVAL_FILE},
    split=None,
)

def format_chat(example):
    """
    Apply Qwen2.5 chat template to the `messages` field.
    Returns a dict with a single `text` key containing the fully formatted string.
    Unsloth's SFTTrainer will tokenise this key directly.
    """
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize         = False,
        add_generation_prompt = False,
    )
    return {"text": text}

train_ds = raw["train"].map(format_chat, batched=False, remove_columns=raw["train"].column_names)
eval_ds  = raw["eval"].map(format_chat,  batched=False, remove_columns=raw["eval"].column_names)

print(f"Train: {len(train_ds)} examples | Eval: {len(eval_ds)} examples")
print("\nFirst example (truncated):")
print(train_ds[0]["text"][:400], "...")

In [ ]:
# Cell 6 — Train

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_ds,
    eval_dataset    = eval_ds,
    dataset_text_field = "text",
    max_seq_length  = MAX_SEQ_LEN,
    dataset_num_proc = 2,
    packing         = False,    # disable packing — examples vary widely in length
    args = TrainingArguments(
        output_dir                  = OUTPUT_DIR,
        num_train_epochs            = NUM_EPOCHS,
        per_device_train_batch_size = PER_DEVICE_BATCH,
        per_device_eval_batch_size  = PER_DEVICE_BATCH,
        gradient_accumulation_steps = GRAD_ACCUM,
        learning_rate               = LEARNING_RATE,
        lr_scheduler_type           = LR_SCHEDULER,
        warmup_ratio                = WARMUP_RATIO,
        weight_decay                = WEIGHT_DECAY,
        max_grad_norm               = MAX_GRAD_NORM,
        bf16                        = True,
        fp16                        = False,
        optim                       = "adamw_8bit",    # saves a few hundred MB
        logging_steps               = LOGGING_STEPS,
        save_strategy               = "steps",
        save_steps                  = SAVE_STEPS,
        evaluation_strategy         = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        report_to                   = REPORT_TO,
        run_name                    = "wagmi-qwen2.5-1.5b",
        seed                        = 42,
        dataloader_num_workers      = 2,
        dataloader_pin_memory       = True,
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"\nTraining complete. Runtime: {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"Train loss: {trainer_stats.metrics['train_loss']:.4f}")

In [ ]:
# Cell 7 — Save adapter + optional push to Hub

# Save LoRA adapter locally
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to ./{OUTPUT_DIR}")

if PUSH_TO_HUB:
    if not HF_TOKEN:
        raise ValueError("HF_TOKEN is not set. Add it in the Space environment variables.")

    model.push_to_hub(
        HUB_MODEL_ID,
        token = HF_TOKEN,
        private = True,
    )
    tokenizer.push_to_hub(
        HUB_MODEL_ID,
        token = HF_TOKEN,
        private = True,
    )
    print(f"Adapter pushed to https://huggingface.co/{HUB_MODEL_ID}")

# Quick inference sanity-check
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system",    "content": "Tu es Wagmi, le watchdog de Deal ex Machina. Reponds de maniere factuelle, concise, sans invention."},
    {"role": "user",      "content": "Qui est Jean-Baptiste Dezard ?"},
]
inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize              = True,
    add_generation_prompt = True,
    return_tensors        = "pt",
).to(model.device)

outputs = model.generate(
    input_ids    = inputs,
    max_new_tokens = 200,
    temperature  = 0.3,
    do_sample    = True,
)
response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print("\n--- Sanity-check response ---")
print(response)